In [2]:
!pip install --upgrade pydantic langchain langchain-core langchain-community pypdf

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ------------------------- -------------- 1.3/2.1 MB 7.3 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 6.7 MB/s  0:00:00
   ---------------------------------------- 0.0/549.1 kB ? eta -:--:--
   ---------------------------------------- 549.1/549.1 kB 7.7 MB/s  0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------------------------- -------- 1.8/2.4 MB 8.9 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 8.7 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 9.7 MB/s  0:00:00

   ----------------------------------------  0/14 [pypdf]
   ----------------------------------------  0/14 [pypdf]
   ----------------------------------------  0/14 [pypdf]
   ------------------

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.0.2 requires pydantic<=2.12.4,>=2.11.10, but you have pydantic 2.13.4 which is incompatible.


In [4]:
!pip install regex

Defaulting to user installation because normal site-packages is not writeable


In [5]:
# from langchain_community.document_loaders import PyPDFLoader
from pypdf import PyPDFLoader
def read_pdf(file_path: str):
    """
    Loads a PDF file and splits it into individual pages.
    """
    loader = PyPDFLoader(file_path)
    pages = loader.load()
    
    print(f"Successfully loaded {len(pages)} pages from the PDF.")
    
    # Example: Print the content of the first page
    if pages:
        print("\n--- First Page Preview ---")
        print(pages[0].page_content[:500]) # First 500 characters
        
    return pages

# Usage:
# pages = read_pdf("sample_lecture.pdf")

ImportError: cannot import name 'PyPDFLoader' from 'pypdf' (C:\Users\DELL\AppData\Roaming\Python\Python314\site-packages\pypdf\__init__.py)

In [6]:
import pypdf

In [7]:
pypdf.PdfReader("rag_sample_document.pdf")

In [8]:
from pypdf import PdfReader

def read_pdf_native(file_path: str):
    """
    Opens a PDF file and extracts all text content page by page.
    """
    reader = PdfReader(file_path)
    extracted_text = []
    
    print(f"Total pages found: {len(reader.pages)}")
    
    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:  # Ensure the page isn't empty/an image
            extracted_text.append(text)
            
    # Combine all pages into one continuous string or keep them partitioned
    full_text = "\n".join(extracted_text)
    print(f"Successfully extracted {len(full_text)} characters.")
    
    # Preview the beginning of the document
    print("\n--- Document Preview ---")
    print(full_text[:500])
    
    return extracted_text

# Usage:
# pages_text = read_pdf_native("sample_lecture.pdf")

In [38]:
sentences = read_pdf_native("rag_sample_document.pdf")

Total pages found: 1
Successfully extracted 2126 characters.

--- Document Preview ---
An Introduction to Retrieval-Augmented Generation
(RAG)
Document ID: TECH-RAG-2026-001 | Classification: Public
Retrieval-Augmented Generation (RAG) is an advanced architectural framework designed to optimize the
output of Large Language Models (LLMs) by sourcing authoritative data from external knowledge bases
before generating a response. Traditional language models rely solely on internal parametric knowledge
acquired  during  their  initial  training  phases,  which  often  renders  them  su


In [10]:
from sentence_transformers import SentenceTransformer
from openai import OpenAI

def generate_embeddings_native(pages_text):
    # 1. Simple sentence/paragraph splitting logic for students
    sentences = []
    for page in pages_text:
        # Splitting by period as a simple way to isolate sentences for the demo
        chunks = [s.strip() for s in page.split('.') if s.strip()]
        sentences.extend(chunks)
    print(f"Created {len(sentences)} text chunks/sentences.")

    # 2. MiniLM Embeddings (Local via SentenceTransformers)
    # Returns a numpy array of embeddings
    minilm_model = SentenceTransformer("all-MiniLM-L6-v2")
    minilm_vectors = minilm_model.encode(sentences)
    print(f"Generated MiniLM vectors. Shape: {minilm_vectors.shape}") # Expecting (N, 384)

    # 3. OpenAI Embeddings (Cloud via official OpenAI SDK)
    # Automatically picks up OPENAI_API_KEY from environment variables
    client = OpenAI()
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=sentences
    )
    # Extract the vectors from the API response
    openai_vectors = [data.embedding for data in response.data]
    import numpy as np
    openai_vectors = np.array(openai_vectors, dtype=np.float32)
    print(f"Generated OpenAI vectors. Shape: {openai_vectors.shape}") # Expecting (N, 1536)

    return sentences, minilm_vectors, openai_vectors

# Usage:
# sentences, minilm_vecs, openai_vecs = generate_embeddings_native(pages_text)

In [11]:
minilm_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\DELL\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
minilm_model.encode('Hi, how are you?')

array([ 2.24338621e-02,  9.20829072e-04,  9.11921188e-02,  7.59619623e-02,
       -1.83316078e-02, -5.42845093e-02,  4.53048088e-02,  1.76668720e-04,
       -7.53027499e-02,  1.15894340e-02, -3.04564219e-02,  1.14464285e-02,
       -2.41077319e-02, -3.15519348e-02,  5.50617948e-02, -2.15418003e-02,
        7.76083916e-02, -9.29088816e-02, -1.27002046e-01,  2.02551875e-02,
       -4.81490903e-02,  2.43427809e-02,  2.25247350e-02,  6.76649213e-02,
       -5.08730821e-02, -4.53872122e-02,  1.37296338e-02,  3.04293465e-02,
       -2.76084226e-02, -7.47584850e-02, -4.39387225e-02,  6.88913167e-02,
       -8.32157508e-02, -3.05768801e-03, -2.10134475e-03,  3.14817615e-02,
       -4.76175435e-02, -1.48482889e-01, -7.76438159e-04, -1.88522656e-02,
        2.49177162e-02, -2.04797667e-02, -1.71083398e-02, -1.30370762e-02,
        9.97296199e-02, -8.27247873e-02,  2.66672038e-02,  5.61491847e-02,
        7.45596960e-02,  3.47898640e-02, -1.11921869e-01, -3.16553377e-02,
       -3.09912022e-03,  

In [13]:
minilm_model.encode('Srimanth')

array([-5.32006174e-02,  2.43133102e-02, -4.19859290e-02, -4.31090854e-02,
       -6.03281632e-02,  5.60517460e-02,  2.82947347e-02, -2.51008049e-02,
       -3.30025032e-02,  3.36207002e-02,  1.89567339e-02, -4.35659029e-02,
        5.07369041e-02, -5.96585777e-03, -1.75010394e-02, -3.30424756e-02,
       -3.10847089e-02,  7.44885504e-02,  3.05715241e-02, -5.03376089e-02,
       -2.47004665e-02,  3.96528058e-02,  7.84301385e-03, -1.59166958e-02,
        4.21368629e-02,  4.51236740e-02,  4.64893766e-02, -1.16575568e-03,
        1.86842438e-02, -3.08659263e-02,  5.43523468e-02, -8.27577785e-02,
       -5.09581044e-02,  4.76187617e-02, -6.87683523e-02,  6.44655200e-03,
       -3.36348452e-02,  4.57626693e-02,  7.03712106e-02,  3.12163588e-02,
       -1.88562497e-02, -7.03890473e-02,  3.96234170e-02, -6.39911741e-02,
        1.35178324e-02, -2.22418960e-02, -9.37593281e-02, -9.96582955e-03,
       -1.65926833e-02, -2.41517685e-02, -3.27260867e-02, -1.45415298e-03,
       -2.80896667e-02,  

In [17]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

True

In [20]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [21]:
client = OpenAI()
response = client.embeddings.create(
        model="text-embedding-3-small",
        input=["Hi, there!"]
    )

In [28]:
len(response.data[0].embedding)

1536

In [ ]:
import faiss
import numpy as np

def build_faiss_retriever(sentences, vectors, query_text, minilm_model, k=2):
    """
    Creates a native FAISS index, adds vectors, and simulates a retriever interface.
    """
    # 1. Initialize the FAISS Index (IndexFlatL2 calculates standard Euclidean distance)
    dimension = vectors.shape[1]  # 384 for MiniLM
    index = faiss.IndexFlatL2(dimension)
    
    # 2. Add vectors to the index (FAISS expects float32 numpy arrays)
    vectors_f32 = vectors.astype('float32')
    index.add(vectors_f32)
    print(f"Indexed {index.ntotal} vectors into FAISS.")

    # --- Simulated "as_retriever" / Retrieval Step ---
    print(f"\n[Retriever] Searching for: '{query_text}'")
    
    # Turn the query text into the same vector space
    query_vector = minilm_model.encode([query_text]).astype('float32')
    
    # Search the index: 'k' is the number of nearest neighbors we want back
    # distances = similarity scores, indices = the position mapping back to our sentences list
    distances, indices = index.search(query_vector, k)

    # Display results mapping the index back to the original text strings
    print(f"\nTop {k} retrieved matches:")
    for rank, idx in enumerate(indices[0]):
        print(f"\n--- Match {rank + 1} (FAISS Index: {idx}) ---")
        print(f"Distance/Score: {distances[0][rank]:.4f}")
        print(f"Text: {sentences[idx]}")

    return index

# Usage:
# index = build_faiss_retriever(sentences, minilm_vecs, "What is the query?", minilm_model)

In [31]:
!pip install langchain_huggingface

Defaulting to user installation because normal site-packages is not writeable


In [33]:
!pip install --upgrade langchain-core langchain-huggingface

Defaulting to user installation because normal site-packages is not writeable
  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
Using cached langchain_huggingface-1.2.2-py3-none-any.whl (31 kB)
  Attempting uninstall: langchain-huggingface
    Found existing installation: langchain-huggingface 1.1.0
    Uninstalling langchain-huggingface-1.1.0:
      Successfully uninstalled langchain-huggingface-1.1.0


In [39]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings, OpenAIEmbeddings


def store_and_retrieve_with_langchain(sentences, query_text: str, use_openai: bool = False):
    """
    Initializes FAISS using LangChain's built-in embedding models,
    stores raw text strings, and exposes a retriever.
    """
    
    # 1. Choose and initialize the LangChain embedding model
    if use_openai:
        print("Initializing LangChain OpenAI Embeddings...")
        # Requires OPENAI_API_KEY environment variable set
        embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    else:
        print("Initializing LangChain HuggingFace (MiniLM) Embeddings...")
        # Downloads and runs the model locally
        embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    # 2. Build the FAISS InStoreDB directly from the raw strings
    print(f"Creating FAISS vector database from {len(sentences)} sentences...")
    vector_db = FAISS.from_texts(
        texts=sentences,
        embedding=embeddings
    )
    print("FAISS database initialized successfully.")

    # 3. Convert the vector database into a retriever interface
    # search_kwargs={"k": 2} retrieves the top 2 most mathematically similar matches
    retriever = vector_db.as_retriever(search_kwargs={"k": 2})

    # 4. Execute the retrieval step
    print(f"\n[Retriever] Querying for: '{query_text}'")
    retrieved_docs = retriever.invoke(query_text)

    # Display the results for your class
    print(f"\nTop retrieved matches:")
    for i, doc in enumerate(retrieved_docs, 1):
        print(f"\n--- Match {i} ---")
        print(f"Content: {doc.page_content}")

    return vector_db, retriever

# Usage for MiniLM:
db, retriever = store_and_retrieve_with_langchain(sentences, "How to implement RAG?", use_openai=False)

# Usage for OpenAI:
# db, retriever = store_and_retrieve_with_langchain(sentences, "What is the text about?", use_openai=True)

Initializing LangChain HuggingFace (MiniLM) Embeddings...


C:\Users\DELL\AppData\Local\Temp\ipykernel_9680\2852189901.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Creating FAISS vector database from 1 sentences...
FAISS database initialized successfully.

[Retriever] Querying for: 'How to implement RAG?'

Top retrieved matches:

--- Match 1 ---
Content: An Introduction to Retrieval-Augmented Generation
(RAG)
Document ID: TECH-RAG-2026-001 | Classification: Public
Retrieval-Augmented Generation (RAG) is an advanced architectural framework designed to optimize the
output of Large Language Models (LLMs) by sourcing authoritative data from external knowledge bases
before generating a response. Traditional language models rely solely on internal parametric knowledge
acquired  during  their  initial  training  phases,  which  often  renders  them  susceptible  to  factual
hallucinations and outdated references. By decoupling the static intelligence of the model from dynamic,
domain-specific information repositories, RAG systems ensure that the synthesized insights are both
contextually accurate and verifiably grounded in real-time enterprise data. 
Th